# Campaign Analysis

This notebook reads the local `results.tsv` and `campaigns.tsv` snapshots, plots validation progress for all campaigns in one figure, and summarizes the one-shot final test result stored at the campaign level.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path('.')
RESULTS_PATH = ROOT / 'results.tsv'
CAMPAIGNS_PATH = ROOT / 'campaigns.tsv'
OUTPUT_PATH = ROOT / 'progress.png'

RESULT_COLUMNS = [
    'campaign_id', 'paper_label', 'experiment_num', 'commit', 'val_ap', 'params_k',
    'status', 'short_caption', 'description', 'branch', 'log_path', 'artifact_path', 'timestamp',
]
CAMPAIGN_COLUMNS = [
    'campaign_id', 'paper_label', 'paper_url', 'baseline_ref', 'start_branch',
    'target_experiments', 'created_at', 'notes', 'final_test_experiment_num',
    'final_test_commit', 'final_test_ap', 'final_test_artifact_path', 'final_test_timestamp',
]


In [ ]:
def load_tsv(path: Path, columns: list[str]) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame(columns=columns)

    frame = pd.read_csv(path, sep='\t')
    for column in columns:
        if column not in frame.columns:
            frame[column] = pd.NA
    return frame[columns].copy()

results = load_tsv(RESULTS_PATH, RESULT_COLUMNS)
campaigns = load_tsv(CAMPAIGNS_PATH, CAMPAIGN_COLUMNS)

if not results.empty:
    results['experiment_num'] = pd.to_numeric(results['experiment_num'], errors='coerce')
    results['val_ap'] = pd.to_numeric(results['val_ap'], errors='coerce')
    results['params_k'] = pd.to_numeric(results['params_k'], errors='coerce')
    results = results.dropna(subset=['experiment_num', 'val_ap']).copy()
    results['experiment_num'] = results['experiment_num'].astype(int)
    results = results.sort_values(['campaign_id', 'experiment_num']).reset_index(drop=True)

if not campaigns.empty:
    campaigns['target_experiments'] = pd.to_numeric(campaigns['target_experiments'], errors='coerce')
    campaigns['final_test_experiment_num'] = pd.to_numeric(campaigns['final_test_experiment_num'], errors='coerce')
    campaigns['final_test_ap'] = pd.to_numeric(campaigns['final_test_ap'], errors='coerce')

print(f'Result rows: {len(results)}')
print(f'Campaign rows: {len(campaigns)}')


In [ ]:
if results.empty:
    summary = campaigns.copy()
    if summary.empty:
        summary = pd.DataFrame(columns=[
            'campaign_id', 'paper_label', 'experiments', 'best_val_ap', 'final_test_ap'
        ])
    else:
        summary = summary.assign(experiments=0, best_val_ap=pd.NA)
        summary = summary[[
            'campaign_id', 'paper_label', 'experiments', 'best_val_ap', 'final_test_ap',
            'final_test_experiment_num', 'final_test_commit', 'target_experiments', 'notes',
        ]]
else:
    result_summary = results.groupby('campaign_id', as_index=False).agg(
        paper_label=('paper_label', 'last'),
        experiments=('experiment_num', 'max'),
        best_val_ap=('val_ap', 'max'),
        latest_val_ap=('val_ap', 'last'),
        latest_caption=('short_caption', 'last'),
    )

    if campaigns.empty:
        summary = result_summary.copy()
        summary['final_test_ap'] = pd.NA
        summary['final_test_experiment_num'] = pd.NA
        summary['final_test_commit'] = pd.NA
        summary['target_experiments'] = pd.NA
        summary['notes'] = pd.NA
    else:
        summary = campaigns.merge(result_summary, on=['campaign_id', 'paper_label'], how='outer')

    summary = summary[[
        'campaign_id', 'paper_label', 'experiments', 'best_val_ap', 'latest_val_ap',
        'latest_caption', 'final_test_ap', 'final_test_experiment_num',
        'final_test_commit', 'target_experiments', 'notes',
    ]].sort_values(['best_val_ap', 'campaign_id'], ascending=[False, True], na_position='last')

display(summary)


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 7))

if results.empty:
    ax.text(0.5, 0.5, 'No experiment results logged yet.', ha='center', va='center', fontsize=14)
    ax.set_axis_off()
else:
    campaign_ids = list(dict.fromkeys(results['campaign_id']))
    cmap = plt.get_cmap('tab10')
    color_map = {campaign_id: cmap(index % 10) for index, campaign_id in enumerate(campaign_ids)}
    edge_map = {'keep': 'black', 'discard': 'white', 'crash': '#d62728'}

    for campaign_id in campaign_ids:
        campaign_df = results[results['campaign_id'] == campaign_id].copy()
        campaign_df['running_best_val_ap'] = campaign_df['val_ap'].cummax()
        color = color_map[campaign_id]
        label = campaign_df['paper_label'].iloc[-1] or campaign_id

        ax.plot(
            campaign_df['experiment_num'],
            campaign_df['running_best_val_ap'],
            color=color,
            linewidth=2.0,
            alpha=0.9,
            label=label,
        )

        ax.scatter(
            campaign_df['experiment_num'],
            campaign_df['val_ap'],
            s=70,
            color=color,
            edgecolors=[edge_map.get(status, 'black') for status in campaign_df['status']],
            linewidths=1.2,
            alpha=0.9,
        )

        kept = campaign_df[campaign_df['status'] == 'keep']
        if not kept.empty:
            last_kept = kept.iloc[-1]
            ax.annotate(
                str(last_kept['short_caption']),
                (last_kept['experiment_num'], last_kept['val_ap']),
                textcoords='offset points',
                xytext=(6, 6),
                fontsize=8,
                color=color,
            )

    ax.set_title('Validation Progress Across Campaigns')
    ax.set_xlabel('Experiment Number')
    ax.set_ylabel('Validation AP')
    ax.legend(loc='best', frameon=True)

fig.tight_layout()
fig.savefig(OUTPUT_PATH, dpi=200, bbox_inches='tight')
print(f'Saved {OUTPUT_PATH}')
